In [6]:
import sys
!{sys.executable} -m pip install pyserial

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import serial
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time
import csv

# -------------------------
# Configuration
# -------------------------
COM_PORT = 'COM3'       # Arduino port
BAUD_RATE = 115200        # From Arduino code
DURATION = 60           # Total duration in seconds
INTERVAL = 1            # Time between readings in seconds
CSV_FILENAME = 'arduino_data.csv'

# -------------------------
# Open Serial Connection
# -------------------------
ser = serial.Serial(COM_PORT, BAUD_RATE, timeout=1)
time.sleep(2)  # Wait for Arduino to reset

# -------------------------
# Lists for plotting
# -------------------------
times = []
values = []

# -------------------------
# Open CSV file and write header
# -------------------------
with open(CSV_FILENAME, 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(["Time_ms", "Pressure_Pa"])  # CSV header

    # -------------------------
    # Read Serial & live plot
    # -------------------------
    start_time = time.time()
    try:
        while time.time() - start_time < DURATION:
            line = ser.readline().decode('utf-8').strip()
            if line:
                try:
                    time_ms, value = line.split(',')
                    time_ms = int(time_ms)
                    value = int(value)
                    times.append(time_ms)
                    values.append(value)

                    # Save to CSV
                    csv_writer.writerow([time_ms, value])

                    # Live plot
                    clear_output(wait=True)
                    plt.figure(figsize=(10,5))
                    plt.plot(times, values, '-o', color='blue')
                    plt.xlabel('Time [ms]')
                    plt.ylabel('Pressure Difference [Pa]')
                    plt.title('Arduino Live Data')
                    plt.grid(True)
                    plt.show()
                except:
                    pass  
            time.sleep(INTERVAL)
    finally:
        ser.close()